# PSI 指标：模型与变量稳定性

计算变量分布漂移（PSI）评估模型与变量稳定性的 Python 实现。

In [2]:
import pandas as pd
import numpy as np
import random
import scorecardpy as sc
import warnings 
warnings.filterwarnings("ignore")

In [107]:
print(pd.__version__)
print(np.__version__)

1.2.4
1.24.4


In [89]:
df = pd.read_csv('psi_var_sample.csv')

In [90]:
def calculate_psi(base_list, test_list, bins=10, min_sample=10):
    try:
        base_df = pd.DataFrame(base_list, columns=['score'])
        test_df = pd.DataFrame(test_list, columns=['score']) 
        
        # 1.去除缺失值后，统计两个分布的样本量
        base_notnull_cnt = len(list(base_df['score'].dropna()))
        test_notnull_cnt = len(list(test_df['score'].dropna()))

        # 空分箱
        base_null_cnt = len(base_df) - base_notnull_cnt
        test_null_cnt = len(test_df) - test_notnull_cnt
        
        # 2.最小分箱数
        q_list = []
        if type(bins) == int:
            bin_num = min(bins, int(base_notnull_cnt / min_sample))
            q_list = [x / bin_num for x in range(1, bin_num)]
            break_list = []
            for q in q_list:
                bk = base_df['score'].quantile(q)
                break_list.append(bk)
            break_list = sorted(list(set(break_list))) # 去重复后排序
            score_bin_list = [-np.inf] + break_list + [np.inf]
        else:
            score_bin_list = bins
        
        # 4.统计各分箱内的样本量
        base_cnt_list = [base_null_cnt]
        test_cnt_list = [test_null_cnt]
        bucket_list = ["MISSING"]
        for i in range(len(score_bin_list)-1):
            left  = round(score_bin_list[i+0], 4)
            right = round(score_bin_list[i+1], 4)
            bucket_list.append("(" + str(left) + ',' + str(right) + ']')
            
            base_cnt = base_df[(base_df.score > left) & (base_df.score <= right)].shape[0]
            base_cnt_list.append(base_cnt)
            
            test_cnt = test_df[(test_df.score > left) & (test_df.score <= right)].shape[0]
            test_cnt_list.append(test_cnt)
        
        # 5.汇总统计结果    
        stat_df = pd.DataFrame({"bucket": bucket_list, "base_cnt": base_cnt_list, "test_cnt": test_cnt_list})
        stat_df['base_dist'] = stat_df['base_cnt'] / len(base_df)
        stat_df['test_dist'] = stat_df['test_cnt'] / len(test_df)
        
        def sub_psi(row):
            # 6.计算PSI
            base_list = row['base_dist']
            test_dist = row['test_dist']
            # 处理某分箱内样本量为0的情况
            if base_list == 0 and test_dist == 0:
                return 0
            elif base_list == 0 and test_dist > 0:
                base_list = 1 / base_notnull_cnt   
            elif base_list > 0 and test_dist == 0:
                test_dist = 1 / test_notnull_cnt
                
            return (test_dist - base_list) * np.log(test_dist / base_list)
        
        stat_df['psi'] = stat_df.apply(lambda row: sub_psi(row), axis=1)
        stat_df = stat_df[['bucket', 'base_cnt', 'base_dist', 'test_cnt', 'test_dist', 'psi']]
        psi = stat_df['psi'].sum()
        
    except:
        print('error!!!')
        psi = np.nan 
        stat_df = None
    return psi, stat_df

In [94]:
# df.head()

In [104]:
var = 'LoanAmount'
base = df.loc[df['date']=='2023-05',var]
test = df.loc[df['date']=='2023-06',var]
calculate_psi(base_list=list(base),test_list=list(test))

(0.00844703517336727,
                bucket  base_cnt  base_dist  test_cnt  test_dist       psi
 0             MISSING         0   0.000000         0   0.000000  0.000000
 1       (-inf,9379.2]       197   0.100254       195   0.098385  0.000035
 2    (9379.2,13673.0]       196   0.099746       194   0.097881  0.000035
 3   (13673.0,17698.4]       197   0.100254       175   0.088295  0.001519
 4   (17698.4,22248.6]       196   0.099746       195   0.098385  0.000019
 5   (22248.6,26700.0]       197   0.100254       231   0.116549  0.002454
 6   (26700.0,31560.6]       196   0.099746       210   0.105954  0.000375
 7   (31560.6,36478.4]       196   0.099746       220   0.110999  0.001203
 8   (36478.4,40893.0]       197   0.100254       176   0.088799  0.001390
 9   (40893.0,45088.4]       196   0.099746       177   0.089304  0.001155
 10      (45088.4,inf]       197   0.100254       209   0.105449  0.000262)

In [95]:
def psi_month_calc(train_df:pd.DataFrame, oot_df:pd.DataFrame, col_list:list, dt_name:str):
    """
    描述:逐月计算oot数据集变量的psi
    输入参数:
    :param train_df: train期望DataFrame
    :param oot_df: oot实际DataFrame
    :param col_list: 变量列表
    :param dt_name: 月份变量的名称
    输出:
    :psi_month_table: 变量在oot上逐月的psi, dataframe
    :psi_month_detail_total: 变量在oot上逐月的psi分箱细节, dict
  """
    month_list = sorted(oot_df[dt_name].unique())
    psi_array = []
    psi_month_detail_total = {}
    for mt in month_list:
        sub_df = oot_df.loc[oot_df[dt_name]==mt]
        psi_month_detail_each = []
        col_psi_dict = {}
        for col in col_list:
            psi, stat_df = calculate_psi(base_list=list(train_df[col]),test_list=list(sub_df[col]),bins=20, min_sample=10)
            stat_df['var_name'] = col
            psi_month_detail_each.append(stat_df)
            col_psi_dict[col] = round(psi,6)
        # 所有变量psi分箱情况，便于后续查看
        psi_month_detail_each.append(stat_df)
        psi_month_detail_total[mt] = pd.concat(psi_month_detail_each)
        # oot每月所有变量的psi值
        psi_array.append(col_psi_dict)
    # oot上逐月变量psi汇总表
    psi_month_table = pd.DataFrame(psi_array).T
    psi_month_table.columns = month_list
    return psi_month_table, psi_month_detail_total

In [98]:
df.columns

Index(['LoanAmount', 'MonthlyIncome', 'CreditScore', 'LoanDuration',
       'HistoricalDelinquencies', 'DebtToIncomeRatio', 'Age',
       'EmploymentDuration', 'OtherLoans', 'Homeownership', 'target', 'date',
       'uid'],
      dtype='object')

In [100]:
base = df.loc[df['date']=='2023-05']
test = df.loc[~(df['date']=='2023-05')]
col_list = df.columns.difference(['target','date','uid']).tolist()
psi_month_table, psi_month_detail_total = psi_month_calc(base,test,col_list,'date')

In [105]:
psi_month_table

,2023-06,2023-07,2023-08,2023-09
Age,0.020921,0.014716,0.014840,0.017420
CreditScore,0.020589,0.024871,0.025346,0.019957
DebtToIncomeRatio,0.016265,0.020026,0.020980,0.009611
EmploymentDuration,0.023678,0.023545,0.019933,0.023306
HistoricalDelinquencies,0.001615,0.002388,0.008223,0.007318
Homeownership,0.000000,0.000000,0.000000,0.000000
LoanAmount,0.011181,0.009834,0.024782,0.012608
LoanDuration,0.014657,0.015265,0.009324,0.011580
MonthlyIncome,0.022671,0.032145,0.024885,0.029975
OtherLoans,0.006652,0.001244,0.005675,0.002530


In [103]:
psi_month_detail_total['2023-06']

,bucket,base_cnt,base_dist,test_cnt,test_dist,psi,var_name
0,MISSING,0,0.000000,0,0.000000,0.000000,Age
1,"(-inf,22.0]",117,0.059542,117,0.059031,0.000004,Age
2,"(22.0,25.0]",104,0.052926,120,0.060545,0.001025,Age
3,"(25.0,28.0]",104,0.052926,116,0.058527,0.000563,Age
4,"(28.0,30.0]",84,0.042748,73,0.036831,0.000881,Age
...,...,...,...,...,...,...,...
7,"(5.0,6.0]",206,0.104835,204,0.102926,0.000035,OtherLoans
8,"(6.0,7.0]",199,0.101272,214,0.107972,0.000429,OtherLoans
9,"(7.0,8.0]",197,0.100254,219,0.110494,0.000996,OtherLoans
10,"(8.0,9.0]",212,0.107888,204,0.102926,0.000234,OtherLoans
